# Cleantech ReAct Agent Implementation

**Name:** Vaishnavi Tiwari

**Date:** November 25, 2025

---
## Project Goal
This project implements a sophisticated **Retrieval-Augmented Generation (RAG)** agent using the **LlamaIndex** framework. The agent is designed to research cleantech topics by strategically combining:
1. A **local Vector Index** (Part 1).
2. A **ReAct Agent** with multiple tool extensions (Part 2).
3. A rigorous **LLM-as-Judge** evaluation harness (Part 3).

---

In [ ]:
# 1. ENVIRONMENT SETUP AND DEPENDENCY INSTALLATION

!pip install transformers==4.38.2
!pip install sentence-transformers==2.5.1
!pip install llama-index-core llama-index-llms-openai llama-index-embeddings-huggingface
!pip install openai pandas

In [ ]:
# 2. IMPORTS AND GLOBAL CONFIGURATION

import os
import csv
import re
import pandas as pd
import asyncio
import json
import numpy as np

# LlamaIndex Core: Essential components for Agents, RAG, and Tools
from llama_index.core import Document, VectorStoreIndex, StorageContext, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.tools import FunctionTool, QueryEngineTool, ToolMetadata
from llama_index.core.agent import ReActAgent
from llama_index.core.llms import ChatMessage
from llama_index.core.postprocessor import LLMRerank
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.response_synthesizers import get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever

# LlamaIndex Integrations: Specific LLM and Embedding Connectors
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

#  CONFIGURATION & LLM SETUP

# OpenAI API key
os.environ["OPENAI_API_KEY"] = "sk- KEY HIDDEN BY AUTHOR "

# LLM: Primary model for Agent reasoning and RAG synthesis (lower latency)
LLM = OpenAI(model="gpt-4o-mini", temperature=0.1)
Settings.llm = LLM
# LLM JUDGE: Powerful model for rigorous evaluation (high accuracy, low temperature)
LLM_JUDGE = OpenAI(model="gpt-4o", temperature=0.0)

# DATA FILE CONSTANTS
DATA_FILE = "cleantech_media_dataset_v3_2024-10-28.csv"
EVAL_FILE = "cleantech_rag_evaluation_data_2024-09-20.csv"

cleantech_query_engine = None
react_agent = None

print("Configuration complete. Proceeding to Part 1: Indexing.")

##  Part 1: RAG Indexing and Query Engine Setup

This section implements the RAG foundation required by Part 1. It utilizes a **robust CSV parsing function** to accommodate files with varying delimiters or extraneous index columns.

* **Embedding Model:** Uses the **HuggingFace BGE-Small** model for efficient vector creation.
* **Chunking Strategy:** Employs SentenceSplitter with a large chunk size to maintain **document context integrity** for complex cleantech articles.
* **Two Core Tools:** The resulting cleantech_query_engine inherently includes the **retriever** and the **response synthesizer (summarizer)**, as required. A **LLMRerank postprocessor** is added to boost retrieval accuracy by reranking the top 50 candidates.

In [ ]:
# PART 1 CODE: RAG INDEXING FUNCTION

def create_cleantech_index(data_path):
    """Loads, cleans, indexes the Cleantech data, and sets up the RAG Query Engine."""
    global cleantech_query_engine

    delimiters = [',', ';', '\t']
    df = None
    EXPECTED_FINAL_COLUMNS = 6

    # Robust CSV parsing logic to handle index columns or varying delimiters
    for delimiter in delimiters:
        try:
            df = pd.read_csv(data_path, engine='python', quoting=csv.QUOTE_NONE, delimiter=delimiter, on_bad_lines='skip')
            if df.shape[1] >= EXPECTED_FINAL_COLUMNS:
                break
            df = None
        except:
            pass

    if df is None:
        print("FATAL: Could not read CSV. Cannot proceed.")
        return False

    # Explicit Column Handling (handles 7-column files with an unnamed index)
    expected_names = ['title', 'date', 'author', 'content', 'domain', 'url']
    num_cols = df.shape[1]
    if num_cols == 7:
        df.columns = ['index_to_drop'] + expected_names
        df = df.drop(columns=['index_to_drop'])
    elif num_cols == 6:
        df.columns = expected_names

    df = df.dropna(subset=['content']).reset_index(drop=True)
    if 'content' not in df.columns:
         print("ERROR: 'content' column not found or mislabeled.")
         return False

    # Creating LlamaIndex Documents
    documents = []
    for index, row in df.iterrows():
        url = row.get('url') if pd.notna(row.get('url')) else ""
        doc = Document(
            text=f"Content: {row.get('content', str(row.to_dict()))}",
            metadata={"source": row.get('domain'), "title": row.get('title'), "date": row.get('date'), "article_url": url}
        )
        documents.append(doc)

    # Chunking and Indexing
    parser = SentenceSplitter(chunk_size=3072, chunk_overlap=20)
    nodes = parser.get_nodes_from_documents(documents)
    embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
    storage_context = StorageContext.from_defaults()

    print(f"Creating Vector Index with {len(nodes)} chunks...")
    index = VectorStoreIndex(nodes, storage_context=storage_context, embed_model=embed_model)
    print("Vector Index created successfully.")

    # RAG Query Engine Configuration with LLMRerank postprocessor
    response_synthesizer = get_response_synthesizer(llm=LLM, response_mode="refine")
    rerank_postprocessor = LLMRerank(top_n=5, llm=LLM)
    cleantech_retriever = VectorIndexRetriever(index=index, similarity_top_k=50)

    cleantech_query_engine = RetrieverQueryEngine(
        retriever=cleantech_retriever,
        response_synthesizer=response_synthesizer,
        node_postprocessors=[rerank_postprocessor]
    )
    print("Query Engine defined.")
    return True

# Executing Part 1
create_cleantech_index(DATA_FILE)

## Part 2: ReAct Agent Assembly and Tool Extensions

This section implements the Agentic AI and the required extensions. The agent uses the **ReAct (Reasoning and Acting)** framework for tool orchestration. We implemented **three powerful extensions** written from scratch, exceeding the minimum requirement of two.

The Agent's system prompt enforces a **strict, hierarchical workflow** to ensure efficient tool selection:
1.  **Local RAG (Internal Data)** $\rightarrow$
2.  **Web Search (External Data)** $\rightarrow$
3.  **Reasoning (Logic)** $\rightarrow$
4.  **Safety Check (Guardrail)**

In [ ]:
#PART 2 CODE: TOOL DEFINITIONS AND AGENT ASSEMBLY

def assemble_agent():
    """Assembles the ReAct agent with all four required tools."""

    global cleantech_query_engine, react_agent
    if cleantech_query_engine is None:
        return False

    # 1. RAG Tool (The core retriever tool)
    cleantech_tool = QueryEngineTool(
        query_engine=cleantech_query_engine,
        metadata=ToolMetadata(name="cleantech_media_search", description=("Provides historical information on renewable energy, EVs, and policy from the Cleantech media dataset. Use this first for factual cleantech questions.")),
    )

    # 2. Reasoning/Calculation Tool (Extension 1: For complex logic)
    def reasoning_tool_func(query: str) -> str:
        """Performs complex logical reasoning, synthesis, comparison, or mathematical calculations."""
        system_prompt = ("You are an expert logical reasoning and calculation engine. Do not search for information.")
        messages = [ChatMessage(role="system", content=system_prompt), ChatMessage(role="user", content=f"Perform the requested reasoning/calculation: {query}"),]
        response = LLM_JUDGE.chat(messages)
        return str(response)

    reasoning_tool = FunctionTool.from_defaults(
        fn=reasoning_tool_func,
        name="reasoning_tool",
        description=("Use this tool for complex logical reasoning, synthesis, comparison, or mathematical calculations."),
    )

    # 3. WEB SEARCH TOOL (Extension 2: For real-time, external data)
    def web_search_tool_func(query: str) -> str:
        """Performs a real-time web search for current or out-of-scope information."""
        # NOTE: This relies on the Google Search Tool being available in the Colab environment
        search_results = google.search(queries=[query])
        return str(search_results)

    web_search_tool = FunctionTool.from_defaults(
        fn=web_search_tool_func,
        name="web_search_tool",
        description=("Performs a real-time search on the internet. Use this **only** for questions requiring current data, general knowledge, or when the cleantech_media_search tool fails."),
    )

    # 4. RESPONSE GUARDRAILING TOOL (Extension 3: Ethical Check)
    def safety_check_tool_func(response_text: str, original_query: str) -> str:
        """Checks the generated response for safety, toxicity, and adherence to the cleantech topic."""
        guardrail_prompt = (
            f"Review the following final response for safety, toxicity, and adherence to the Cleantech topic. "
            f"Original Query: '{original_query}' | Response to Check: '{response_text}'. "
            f"If the response is safe, on-topic, and non-toxic, return ONLY 'PASS'. Otherwise, revise the response to be safe and appropriate."
        )
        messages = [ChatMessage(role="system", content="You are a dedicated Guardrail LLM."), ChatMessage(role="user", content=guardrail_prompt),]
        guardrail_response = LLM_JUDGE.chat(messages)
        return str(guardrail_response).strip()

    safety_check_tool = FunctionTool.from_defaults(
        fn=safety_check_tool_func,
        name="safety_check_tool",
        description=("Used to perform a final safety, toxicity, and topic adherence check on a response."),
    )

    # Assembling the ReAct Agent
    system_prompt = (
        "You are a sophisticated Cleantech Research Agent. Your workflow is: "
        "1. **Always** check `cleantech_media_search` first. "
        "2. If RAG fails, use `web_search_tool`. "
        "3. Use `reasoning_tool` for complex logic or math. "
        "4. **Always** pass your final answer to the `safety_check_tool` before final output."
    )
    chat_messages = [ChatMessage(role="system", content=system_prompt)]

    react_agent = ReActAgent(
        tools=[cleantech_tool, reasoning_tool, web_search_tool, safety_check_tool],
        llm=LLM,
        verbose=False,
        chat_history=chat_messages,
    )
    print("Cleantech ReAct Agent assembled with all four tools.")

    return True

# Executing Part 2
assemble_agent()

## Part 3: Rigorous Evaluation Harness

This section implements the evaluation phase required by Part 3.

### LLM-as-Judge Implementation
We use the **LLM-as-Judge** methodology, which is a state-of-the-art method for RAG system evaluation. GPT-4o is used to score the agent's response against the 'Desired Output' based on three key metrics:
1.  **Faithfulness:** Measures grounding in retrieved facts (hallucination prevention).
2.  **Relevance:** Measures topic adherence and answer-completeness to the original question.
3.  **Completeness:** Measures how well the response covers the required information.

### Execution
The agent execution uses a defensive try...except block to generate a report (cleantech_evaluation_results.csv) even if environment-specific execution errors occur, providing complete trace data for the final report.

In [ ]:
# PART 3 CODE: EVALUATION FUNCTIONS

def load_evaluation_data(file_path):
    """Loads evaluation data from the provided CSV file."""
    try:
        df = pd.read_csv(file_path, sep=';')
        df = df.rename(columns={'question': 'question', 'answer': 'desired_output', 'article_url': 'expected_articles'})
        df = df[['question', 'desired_output', 'expected_articles']].dropna().reset_index(drop=True)
        print(f"Loaded {len(df)} evaluation questions from {file_path}.")
        return df.to_dict('records')
    except Exception as e:
        print(f"ERROR reading evaluation file: {e}")
        return []

async def llm_as_judge(question, agent_response, ground_truth):
    """Uses LLM_JUDGE (GPT-4o) to score the agent's response."""

    judge_prompt = (
        f"You are an expert RAG evaluation judge. Your task is to score the Agent's Response "
        f"against the Ground Truth and Question. Output a JSON object with three scores (0 or 1).\n"
        f"Scores:\n1. relevance\n2. faithfulness\n3. completeness\n"
        f"Question: {question} | Ground Truth: {ground_truth} | Agent's Response: {agent_response}\n"
        f"Return ONLY the JSON object. Example: {{\"relevance\": 1, \"faithfulness\": 1, \"completeness\": 1}}"
    )
    messages = [ChatMessage(role="system", content="You are a dedicated RAG Judge LLM."), ChatMessage(role="user", content=judge_prompt),]
    response = LLM_JUDGE.chat(messages)

    try:
        json_match = re.search(r'\{.*\}', str(response), re.DOTALL)
        if json_match:
            data = json.loads(json_match.group(0))
            data['relevance'] = int(data.get('relevance', 0))
            data['faithfulness'] = int(data.get('faithfulness', 0))
            data['completeness'] = int(data.get('completeness', 0))
            return data
        return None
    except Exception:
        return None

async def run_full_evaluation(eval_data):
    """Executes the agent for all evaluation questions and calculates final metrics."""

    global react_agent
    results = []

    print("\n\n*** STARTING AGENT EXECUTION ***")

    for i, item in enumerate(eval_data):
        question = item["question"]
        ground_truth = item["desired_output"]

        # 1. Executing the Agent (Defensive call)
        try:
            agent_response_obj = react_agent.run(question)

            # Defensive Extraction of the final answer text
            try:
                agent_answer = str(agent_response_obj.response)
            except AttributeError:
                agent_answer = str(agent_response_obj)

            is_rag_used = "cleantech_media_search" in str(agent_response_obj)

        except Exception as e:
            agent_answer = f"AGENT ERROR: {type(e).__name__}: {e}"
            is_rag_used = False

        # 2. LLM as Judge (Score the response)
        judge_scores = await llm_as_judge(question, agent_answer, ground_truth)

        # 3. Storing Results
        results.append({
            "question": question,
            "ground_truth": ground_truth,
            "agent_response": agent_answer,
            "retrieval_status": "RAG Used" if is_rag_used else "Web/Reasoning Only",
            "judge_scores": judge_scores
        })

    # Final Metric Calculation
    valid_scores = [r['judge_scores'] for r in results if r['judge_scores'] is not None]

    if valid_scores:
        total_relevance = sum(s.get('relevance', 0) for s in valid_scores)
        total_faithfulness = sum(s.get('faithfulness', 0) for s in valid_scores)
        total_completeness = sum(s.get('completeness', 0) for s in valid_scores)

        num_valid = len(valid_scores)

        print("\nFinal Evaluation Metrics ")
        print(f"Total Questions Evaluated by Judge: {num_valid} / {len(eval_data)}")
        print(f"Average Relevance (0-1): {total_relevance/num_valid:.4f}")
        print(f"Average Faithfulness (0-1): {total_faithfulness/num_valid:.4f}")
        print(f"Average Completeness (0-1): {total_completeness/num_valid:.4f}")
    else:
        print("No valid scores could be generated by the LLM Judge. (Execution likely resulted in AGENT ERRORs.)")

    return results

# FINAL EXECUTION BLOCK

async def execute_project():
    """Runs the entire project: Part 1, Part 2, and Part 3 sequentially."""

    print("\n\nSTARTING PROJECT EXECUTION ")

    eval_data = load_evaluation_data(EVAL_FILE)

    if not eval_data:
        print("Evaluation cancelled due to data loading error.")
        return

    final_results = await run_full_evaluation(eval_data)

    # Output results to CSV
    results_df = pd.DataFrame(final_results)
    output_filename = "cleantech_evaluation_results.csv"
    results_df.to_csv(output_filename, index=False)
    print(f"\nEvaluation complete. Detailed results saved to: {output_filename}")

    # Print sample for immediate review
    print("\nSample of Detailed Results (First 3):")
    for i, res in enumerate(final_results[:3]):
        print(f"\nQ{i+1}: {res['question']}")
        print(f"  -> Agent Response: {res['agent_response'][:100]}...")
        print(f"  -> Retrieval Used: {res['retrieval_status']}")
        print(f"  -> Judge Score: {res['judge_scores']}")


# Executing the entire project
await execute_project()